# Demand Predictor — Train on Colab (GPU)

Trains the `DemandPredictor` U-Net (Stage 3 of the completing-router overhaul). It learns to
predict, per layer, **where upcoming nets will route**. Training uses a **variable horizon**:
usually the next k ≤ K nets, sometimes *all* remaining nets — so the same checkpoint serves two
roles at inference:

1. **Next-K lookahead**: route the current net to leave space for the next few nets
   (A* cost = obstacles + λ·demand).
2. **Whole-board demand**: pass every unrouted net's pins to get a full congestion forecast —
   used to warm-start the rip-up router's cost field and to compute the routability/overflow
   score (predicted demand vs. capacity).

Supervised — labels come from boards completed by the rip-up-and-reroute router.

**Before you run:** make sure these files are pushed to GitHub:
`pcb_router/routing/rip_up_router.py`, `pcb_router/models/demand_predictor.py`,
`scripts/train_demand_predictor.py`.

Everything (dataset, checkpoints, training-progress images) is saved to **Google Drive** so it
survives disconnects/resets — see the Drive-mount cell below.

**Runtime:** set Runtime → Change runtime type → GPU.

## 0. Setup

In [ ]:
import os, sys
REPO = '/content/Router'
if not os.path.exists(REPO):
    !git clone https://github.com/Klutzhehe/Router.git $REPO
os.chdir(REPO)
!git pull --ff-only
if REPO not in sys.path:
    sys.path.insert(0, REPO)   # chdir alone does NOT put the repo on sys.path; imports need this
!pip -q install scipy pyyaml
import torch
assert os.path.exists('scripts/train_demand_predictor.py'), 'new files missing - did git pull fetch them?'
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| files OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Everything (dataset, checkpoints, training-progress images) is saved under this Drive folder so
# it survives Colab disconnects/resets. Change the folder name if you want a different project.
DRIVE_DIR = '/content/drive/MyDrive/pcb_router'
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints/viz', exist_ok=True)

DATA_PATH = f'{DRIVE_DIR}/data/demand_dataset.pkl'
CKPT_PATH = f'{DRIVE_DIR}/checkpoints/demand_predictor.pt'
VIZ_DIR   = f'{DRIVE_DIR}/checkpoints/viz'
print('Saving dataset/checkpoints/viz to Google Drive at:', DRIVE_DIR)

## 1. Generate the dataset

Routes boards with the rip-up-and-reroute router and keeps only fully-routed boards (clean labels).
A single stage / small board count is enough to prove the pipeline works, but too thin (and too
narrow) for the model to generalize - it needs to see enough variety of board size, net count,
layer count, congestion, diff pairs, and length-matched nets. This mixes three stages of
increasing difficulty, weighted by how expensive each is to route (the congested stage gets fewer
boards since it's the slowest, but still contributes plenty of samples since it has many nets per
board):

| stage | boards | nets/board | notes |
|---|---|---|---|
| `s10_via_plus_multi_net` | 120 | 3-6 | fast; plain multi-net + via escape, no diff/length constraints |
| `s11_multi_net_multi_layer_diffpairs` | 100 | 4-10 | diff pairs + length matching together |
| `s12_congested_multi_net` | 60 | 8-15 | slow but valuable; heavy congestion, tighter length tolerance |

~280 boards -> roughly ~1,600-1,800 training samples (about 4-5x the earlier single-stage run).
Each stage is generated with a separate call to the SAME `DATA_PATH`, so they accumulate on top of
each other (and of any boards already saved there) rather than overwriting - this is the resumable
dataset generation, so a Colab disconnect partway through only costs boards since the last
checkpoint, not the whole run. Re-running this cell later with higher counts will just add more.

**Generation is now parallel** (`workers=` routing processes; boards that fail to fully route are
discarded after paying full routing cost, so this is close to a linear speedup) and each kept
board logs the running **acceptance rate** — if a hard stage shows e.g. 15%, most wall time is
discarded boards, which is the signal to lower `max_iters`/`astar_max_iterations` rather than wait.

In [ ]:
import os
from scripts.train_demand_predictor import generate_dataset

# (stage, board_count) - counts weighted DOWN for the more expensive/congested stages so total
# generation time stays reasonable while still covering the full difficulty range.
STAGE_PLAN = [
    ('s10_via_plus_multi_net', 120),
    ('s11_multi_net_multi_layer_diffpairs', 100),
    ('s12_congested_multi_net', 60),
]

# Routing is pure-Python and single-threaded; boards are independent, so use every core. (On
# Colab this is typically 2-8 workers depending on runtime type.)
WORKERS = os.cpu_count()

samples = []
for stage_name, count in STAGE_PLAN:
    samples = generate_dataset(
        stage_names=[stage_name],
        boards_per_stage=count,
        out_path=DATA_PATH,   # same file every call -> Drive, resumable/cumulative across stages
        max_iters=8,
        workers=WORKERS,
    )
    print(f'-- after {stage_name}: {len(samples)} boards total --')

print('boards generated:', len(samples))

# Quick check that diff-pair / length-matched nets actually showed up (and therefore that their
# routes are in the labels the predictor will train on).
n_diff = sum(1 for s in samples for n in s['board'].nets if n.is_diff_pair)
n_matched = sum(1 for s in samples for n in s['board'].nets if n.matched_group_id is not None)
print(f'diff-pair nets across dataset: {n_diff}   length-matched nets: {n_matched}')

## 2. Train the predictor

Weighted BCE with **auto-balanced positive weighting** (the neg/pos ratio per batch, clamped to
[1, 20]) — needed because labels now range from very sparse (next-1-net) to dense
(all-remaining-nets full-horizon samples), so no fixed weight fits all. `FULL_FUTURE_PROB`
controls what fraction of samples train on the full horizon; those are what make the checkpoint
usable as a whole-board demand predictor (rip-up warm-start + routability score). Batches pad to
the batch's max size, so no wasted compute on small boards. Streams one line per epoch to the log
(loss, on-route vs. off-route predicted probability, elapsed time). Checkpoints to **Google Drive**
every `SAVE_EVERY` epochs (and at the end), and can **resume** from there if the runtime disconnects.

In [ ]:
import os, time, pickle, torch
from torch.utils.data import DataLoader
from scripts.train_demand_predictor import DemandDataset, collate_pad, demand_loss
from pcb_router.models.demand_predictor import DemandPredictor

# ---- config (edit me) ----
DATA       = DATA_PATH     # Google Drive path, set in the Drive-mount cell above
EPOCHS     = 80
BATCH      = 8
LR         = 2e-3
K          = 3
FULL_FUTURE_PROB = 0.25    # fraction of samples whose future set is ALL remaining nets
                           # (enables whole-board demand prediction: warm-start + routability)
CKPT       = CKPT_PATH     # Google Drive path -> checkpoint survives disconnects
SAVE_EVERY = 10            # checkpoint to Drive every N epochs, plus always at the end
RESUME     = True          # continue from CKPT if it already exists in Drive
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- data + model ----
samples = pickle.load(open(DATA, 'rb'))
ds = DemandDataset(samples, K=K, full_future_prob=FULL_FUTURE_PROB)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate_pad)
model = DemandPredictor().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

start_epoch = 0
if RESUME and os.path.exists(CKPT):
    ckpt = torch.load(CKPT, map_location=device)
    model.load_state_dict(ckpt['model'])
    if 'opt' in ckpt:
        opt.load_state_dict(ckpt['opt'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'resumed from {CKPT} at epoch {start_epoch}', flush=True)

print(f'{len(ds)} samples from {len(samples)} boards | device={device}', flush=True)

# Fixed eval batch for a quick quality metric each epoch (predicted prob ON vs OFF the true routes).
n_eval = min(16, len(ds))
exb, eyb, elmb, esmb = collate_pad([ds[i] for i in range(n_eval)])
exb, eyb, elmb, esmb = exb.to(device), eyb.to(device), elmb.to(device), esmb.to(device)

def save_checkpoint(epoch):
    os.makedirs(os.path.dirname(CKPT) or '.', exist_ok=True)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'K': K,
                'full_future_prob': FULL_FUTURE_PROB, 'epoch': epoch}, CKPT)

# ---- training loop (streams one line per epoch to the log; checkpoints to Drive) ----
t0 = time.time()
for ep in range(start_epoch, EPOCHS):
    model.train(); tot = nb = 0
    for x, y, lm, sm in dl:
        x, y, lm, sm = x.to(device), y.to(device), lm.to(device), sm.to(device)
        loss = demand_loss(model(x), y, lm, sm)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    model.eval()
    with torch.no_grad():
        p = model.predict(exb, elmb)
    on  = p[eyb > 0.5].mean().item()                                  # want this to climb
    off = p[(eyb < 0.5) & (esmb.expand_as(eyb) > 0)].mean().item()    # want this low
    print(f'epoch {ep+1:3d}/{EPOCHS} | loss {tot/nb:.4f} | on-route {on:.3f}  off-route {off:.3f}  '
          f'(gap {on-off:+.3f}) | {time.time()-t0:5.0f}s', flush=True)

    if (ep + 1) % SAVE_EVERY == 0 or ep == EPOCHS - 1:
        save_checkpoint(ep + 1)
        print(f'  checkpoint saved -> {CKPT}', flush=True)

print('done. final checkpoint at', CKPT, flush=True)

## 3. Sanity check — predicted demand vs. actual future routes

On a trained model the **PREDICTED demand** should light up where the **ACTUAL future routes** are
(and along the corridor between the next-K pins). This picks a sample where the current net or one
of the next-K nets is a **differential pair** or **length-matched** net specifically, so you can see
the predictor handling those cases (not just plain single nets). Also saves the figure to Drive.

In [ ]:
import pickle
import matplotlib.pyplot as plt
import torch
from pcb_router.models.demand_predictor import DemandPredictor, MAX_LAYERS
from scripts.train_demand_predictor import DemandDataset, collate_pad

samples = pickle.load(open(DATA_PATH, 'rb'))
K = 3
ds = DemandDataset(samples, K=K)

model = DemandPredictor().cuda()
model.load_state_dict(torch.load(CKPT_PATH, map_location='cuda')['model'])
model.eval()


def find_demo_sample(ds, want='diff'):
    """Scan the dataset for a sample where the current net or a next-K net is a diff-pair
    (want='diff') or length-matched (want='matched') net. Falls back to index 0 if none found."""
    for idx, (bi, i) in enumerate(ds.index):
        s = ds.samples[bi]
        board, order = s['board'], s['order']
        net_ids = order[i:i + 1 + K]
        nets = [n for n in board.nets if n.id in net_ids]
        if want == 'diff' and any(n.is_diff_pair for n in nets):
            return idx, board, order[i]
        if want == 'matched' and any(n.matched_group_id is not None for n in nets):
            return idx, board, order[i]
    return 0, ds.samples[ds.index[0][0]]['board'], ds.samples[ds.index[0][0]]['order'][ds.index[0][1]]


def show_sample(idx, title_extra='', L0=0):
    # Pin channels are now PER-LAYER (see demand_predictor.py): [8:16]=current net pins,
    # [16:24]=next-K nets pins, one sub-channel per layer. Show layer L0's slice of each so all
    # four panels are directly comparable on the same layer.
    x, y, lm = ds[idx]
    xb, yb, lmb, smb = collate_pad([ds[idx]])
    with torch.no_grad():
        p = model.predict(xb.cuda(), lmb.cuda())[0].cpu()

    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    ax[0].imshow(x[MAX_LAYERS + L0]);       ax[0].set_title(f'current net pins (L{L0})')
    ax[1].imshow(x[MAX_LAYERS * 2 + L0]);   ax[1].set_title(f'next-K nets pins (L{L0})')
    ax[2].imshow(p[L0]);                    ax[2].set_title(f'PREDICTED demand (L{L0})')
    ax[3].imshow(y[L0]);                    ax[3].set_title(f'ACTUAL future routes (L{L0})')
    for a in ax:
        a.axis('off')
    fig.suptitle(f'sample {idx}{title_extra}', color='black')
    plt.tight_layout()
    return fig


# Demo 1: a sample touching a differential pair.
idx_diff, board_d, net_id_d = find_demo_sample(ds, want='diff')
fig = show_sample(idx_diff, ' (near a differential pair)')
fig.savefig(f'{VIZ_DIR}/sanity_check_diffpair.png', dpi=100, bbox_inches='tight')
plt.show()

# Demo 2: a sample touching a length-matched net.
idx_matched, board_m, net_id_m = find_demo_sample(ds, want='matched')
fig = show_sample(idx_matched, ' (near a length-matched net)')
fig.savefig(f'{VIZ_DIR}/sanity_check_lengthmatched.png', dpi=100, bbox_inches='tight')
plt.show()